# Lesson 8 (part 2): Sequence models on KKBox

In notebook 06, we turned every currently active KKBox user into a **21-day listening sequence**, with the label churn = 1 if they didn't listen at all in days 22 to 31.
Now we compare four models on the **same test users**:
1. **XGBoost on 21-day totals:** the time order is thrown away.
2. **XGBoost on hand-made time features:** first week vs. last week, recency and trend.
3. **LSTM** and 4. **GRU:** deep learning that reads the raw sequence **day by day**.

At the end, a **shuffle test** checks whether the LSTM really uses the order of the days.

# Setup

**Imports.** The same tools as in Lessons 6 and 7, plus:
- `average_precision_score` for a new metric, **PR-AUC** (explained below);
- `time` to measure how long the networks take to train.

The backend is TensorFlow, as in Lesson 7 (the commented-out line would switch to PyTorch). `keras.utils.set_random_seed(42)` makes the results repeatable.

In [ ]:
import os
# os.environ["KERAS_BACKEND"] = "torch"   # uncomment ONLY if Lesson 7 used the PyTorch backend

import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import keras
from keras import layers
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             confusion_matrix, ConfusionMatrixDisplay)

keras.utils.set_random_seed(42)

**Load the sequences** saved by notebook 06: `raw` (users, 21 days, 7 features), `active` (users, 21 days) and the labels `y`.
`feature_cols` lists the 7 features in the same order as in notebook 06.

In [ ]:
data = np.load("../data/kkbox/sequences.npz")
raw, active, y = data["raw"], data["active"], data["y"]
feature_cols = ["num_25", "num_50", "num_75", "num_985", "num_100", "num_unq", "total_mins"]

print(raw.shape, active.shape, "churn rate:", round(y.mean() * 100, 1), "%")

**Split the users into train, validation and test.**

```
All users
    Test (20%)              the final exam for every model
    Training (80%)          XGBoost learns from all of these
        Train (64%)         the LSTM and GRU learn from these
        Validation (16%)    early stopping for the LSTM and GRU
```

- We split **users**, so each user appears in only one set. If the same user were in train and test, the model could simply recognize them.
- `stratify` keeps the same churn rate in every set.
- We split **index numbers** (`idx`) instead of the data itself, so exactly the same split can be used for the XGBoost tables and for the 3D sequences.
- As in Lesson 7: **XGBoost trains on train + validation** (`idx_train`; it doesn't need a validation set), the **neural networks train on train** (`idx_tr`) and use **validation** (`idx_val`) for early stopping, and **every model is tested on the same test users** (`idx_test`).

In [ ]:
idx = np.arange(len(y))
idx_train, idx_test = train_test_split(idx, test_size=0.2, random_state=42, stratify=y)
idx_tr, idx_val = train_test_split(idx_train, test_size=0.2, random_state=42, stratify=y[idx_train])

print("Train:", len(idx_tr), " Validation:", len(idx_val), " Test:", len(idx_test))

**Evaluation helper, with a new metric: PR-AUC.**
`evaluate(name, y_true, y_prob)` computes the same metrics as in earlier lessons from the churn probabilities, plus one new metric:
- **PR-AUC** (average precision) summarizes the **precision-recall curve**. Sort the users from the highest to the lowest churn risk and walk down the list: PR-AUC measures how "clean" the top of the list is, i.e. how many of the flagged users really churn.
- It focuses on the **churners**, so it is very informative when churners are a minority. A random model scores about the **churn rate** (that's the floor), and a perfect model scores 1.0.
- `zero_division=0` avoids a warning if a model never predicts churn (precision would be 0 divided by 0).

The **lazy model** (predict that everyone stays active) is added by hand: accuracy = the share of test users who stayed, precision, recall and F1 = 0, ROC-AUC = 0.5 (random ranking), and PR-AUC = the churn rate (the floor).

In [ ]:
def evaluate(name, y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    scores = {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
    }
    for key, value in scores.items():
        if key != "Model":
            print(f"{key:9}: {value:.3f}")

    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred),
                           display_labels=["Stayed active", "Churned"]).plot(cmap="Blues")
    plt.title(name)
    plt.show()
    return scores

results = []
churn_rate_test = y[idx_test].mean()
results.append({"Model": "Lazy (predict all stay)", "Accuracy": 1 - churn_rate_test,
                "Precision": 0, "Recall": 0, "F1": 0, "ROC-AUC": 0.5, "PR-AUC": churn_rate_test})

# Part A: XGBoost baselines

**Hand-made features for XGBoost (feature engineering).**
XGBoost needs **one row per user**, not a sequence, so we turn each user's 21 days into numbers.

1. **Totals only** (`total_cols`): the sum of each feature over the 21 days, plus the number of active days. This is what you get by **flattening** each user into one row: the time order is thrown away (listening a lot early and nothing late looks the same as the opposite).
2. **Time features**, a human's attempt to capture time patterns:
   - **first week vs. last week:** sums over days 1 to 7 (`raw[:, :7]`) and days 15 to 21 (`raw[:, 14:]`, because Python counts from 0);
   - the number of active days in the first and the last week;
   - **recency:** days since the user last listened. `active[:, ::-1]` reads each user's days **backwards** (`::-1` reverses the order), and `np.argmax` finds the first 1, i.e. the most recent listening day. If a user listened on day 21, the recency is 0;
   - **trend:** minutes in the last week minus minutes in the first week (negative = listening less).

The big question of this lesson: can an LSTM learn these time patterns **by itself**, from the raw days?

In [ ]:
feats = pd.DataFrame()
for i, col in enumerate(feature_cols):
    feats[f"total_{col}"] = raw[:, :, i].sum(axis=1)
feats["active_days"] = active.sum(axis=1)

total_cols = list(feats.columns)   # "totals only": time order thrown away

for i, col in enumerate(feature_cols):
    feats[f"first7_{col}"] = raw[:, :7, i].sum(axis=1)
    feats[f"last7_{col}"] = raw[:, 14:, i].sum(axis=1)
feats["active_days_first7"] = active[:, :7].sum(axis=1)
feats["active_days_last7"] = active[:, 14:].sum(axis=1)
last_active_day = 21 - np.argmax(active[:, ::-1], axis=1)
feats["days_since_last_active"] = 21 - last_active_day
feats["mins_trend"] = feats["last7_total_mins"] - feats["first7_total_mins"]

print("Totals-only features:", len(total_cols), "| All time features:", feats.shape[1])

**XGBoost on totals only.**
The same settings as the hand-picked XGBoost in Lesson 6. `scale_pos_weight` is the ratio of stayed to churned users in the training users, to handle the imbalance (Lesson 6).
It is trained on `idx_train` (train + validation) and evaluated on the test users.

In [ ]:
ratio = (y[idx_train] == 0).sum() / (y[idx_train] == 1).sum()

xgb_totals = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                           subsample=0.8, colsample_bytree=0.8, scale_pos_weight=ratio,
                           eval_metric="logloss", random_state=42)
xgb_totals.fit(feats.iloc[idx_train][total_cols], y[idx_train])
results.append(evaluate("XGBoost (21-day totals)",
                        y[idx_test], xgb_totals.predict_proba(feats.iloc[idx_test][total_cols])[:, 1]))

**XGBoost on totals + time features.** The same settings, but now with all the features, including first week vs. last week, recency and trend.

In [ ]:
xgb_time = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                         subsample=0.8, colsample_bytree=0.8, scale_pos_weight=ratio,
                         eval_metric="logloss", random_state=42)
xgb_time.fit(feats.iloc[idx_train], y[idx_train])
results.append(evaluate("XGBoost (time features)",
                        y[idx_test], xgb_time.predict_proba(feats.iloc[idx_test])[:, 1]))

*Pending: this explanation is written from the real results, after the first run on the KKBox data.* <!-- PLACEHOLDER_XGB -->

# Part B: Sequence models

**Prepare the sequence input for the neural networks.**
- `np.log1p(raw)` computes log(1 + x) for every value. Listening counts are very **skewed**: most users play a few songs a day, some play hundreds. log1p squeezes big numbers into a small range, so the network trains well, and it keeps 0 as 0 (log(1 + 0) = 0).
  It needs no fitting (unlike a StandardScaler), so there is no leakage risk.
- `active[:, :, None]` adds a new last axis to the active flag, shape (users, 21, 1), so `np.concatenate(..., axis=-1)` can glue it on as an **8th feature** per day. Final shape: `(users, 21, 8)`.
- `y_float`: the same labels as decimals, the format Keras expects for training (same as Lesson 7).
- **Class weights:** the same formula as Lesson 7, computed on the train users.

In [ ]:
X_seq = np.concatenate([np.log1p(raw), active[:, :, None]], axis=-1).astype("float32")
y_float = y.astype("float32")
print("Sequence input shape:", X_seq.shape, "| largest value:", round(float(X_seq.max()), 1))

n_tr = len(idx_tr)
n_churn = y[idx_tr].sum()
class_weight = {0: float(n_tr / (2 * (n_tr - n_churn))), 1: float(n_tr / (2 * n_churn))}
print("Class weights:", {k: round(v, 2) for k, v in class_weight.items()})

## What are RNN, LSTM and GRU?

**Recurrent neural networks (RNN)**
- The MLP in Lesson 7 sees all its inputs **at once** and has no idea of order. If we gave it 21 days, "day 3" and "day 20" would just be two different input columns.
- A **recurrent neural network (RNN)** reads the sequence **one day at a time** and keeps a **memory**, called the **hidden state**, which it updates after each day.
- It's like reading a diary **page by page** and keeping a running impression of the person. After the last page, your impression sums up the whole story, including **what changed recently**.
- The same weights are used for every day: the network applies the same "update rule" at each step.

**LSTM (Long Short-Term Memory)**
- Simple RNNs **forget early days** on long sequences: during training, the error signal shrinks a little at every step back in time, so the early steps barely learn. This is the **vanishing gradient** problem.
- An **LSTM** adds **gates**: small learned switches between 0 and 1 that decide, at each step,
  1. what to **forget** from the memory (forget gate),
  2. what new information to **store** (input gate),
  3. what to **output** from the memory (output gate).
- It also has a separate long-term memory, the **cell state**, through which information can flow from early days almost unchanged.

**GRU (Gated Recurrent Unit)**
- A **GRU** is a simpler cousin of the LSTM, with **2 gates** (update and reset) instead of 3 and no separate cell state, so it has **fewer parameters**.
- It is often just as good, especially on **short sequences** like our 21 days.

**How our models end:** after reading all 21 days, the LSTM (or GRU) hands over its final memory: 32 numbers that summarize the user's listening history. They go into a small Dense layer and a **sigmoid** output, which gives the **churn probability**, exactly like the MLP in Lesson 7.

```
Input            21 days, 8 numbers per day
LSTM or GRU      reads the days in order, keeps a memory of 32 numbers
Dropout(0.3)     switches off 30% of the memory values during training
Dense(16)        ReLU
Dense(1)         sigmoid: the churn probability
```

**Build the LSTM.** `layers.LSTM(32)` reads the 21 days and outputs its final memory of 32 numbers. The rest is the same as in Lesson 7.

**Counting the parameters** (expected total: **5,793**):
- LSTM layer: it has **4** internal parts (the 3 gates plus the "candidate" new memory). Each part has weights for the 8 inputs and for the 32 memory values, plus a bias for each of the 32 units: 4 x (32 x (8 + 32) + 32) = **5,248**
- Dense(16): 32 x 16 + 16 = 528
- Output: 16 x 1 + 1 = 17
- Total: 5,248 + 528 + 17 = **5,793**

In [ ]:
keras.utils.set_random_seed(42)

lstm = keras.Sequential([
    keras.Input(shape=(21, 8)),
    layers.LSTM(32),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
lstm.summary()

**Compile and train the LSTM.** Compile and early stopping work exactly as in Lesson 7. What changed:
- `batch_size=256` (Lesson 7: 32): we now have tens of thousands of users, so bigger batches keep training fast.
- `patience=5` (Lesson 7: 15): each epoch takes longer, and with many more validation users, the validation AUC is much less noisy.
- `epochs=50` is only an upper limit; early stopping ends training sooner and restores the best weights.
- `verbose=2` prints **one line per epoch** (the training and validation loss and AUC), so you can watch the progress.

After training, we print the training time, the number of epochs and the best epoch.

In [ ]:
lstm.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
             loss="binary_crossentropy",
             metrics=[keras.metrics.AUC(name="auc")])

early_stop = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                           patience=5, restore_best_weights=True)

start = time.time()
lstm_history = lstm.fit(X_seq[idx_tr], y_float[idx_tr],
                        validation_data=(X_seq[idx_val], y_float[idx_val]),
                        epochs=50, batch_size=256,
                        class_weight=class_weight,
                        callbacks=[early_stop], verbose=2)
print(f"Training time: {time.time() - start:.0f} seconds")
print("Epochs trained:", len(lstm_history.history["loss"]),
      "| Best epoch:", int(np.argmax(lstm_history.history["val_auc"])) + 1)

**Evaluate the LSTM on the test users.** `.ravel()` flattens the output from shape (users, 1) to (users,), as in Lesson 7.

In [ ]:
results.append(evaluate("LSTM", y[idx_test], lstm.predict(X_seq[idx_test], verbose=0).ravel()))

**Build the GRU.** Only the recurrent layer changes: `layers.GRU(32)` instead of `layers.LSTM(32)`.

**Counting the parameters** (expected total: **4,577**):
- GRU layer: it has **3** internal parts (the 2 gates plus the candidate new memory), with weights for the 8 inputs and the 32 memory values. Keras's GRU uses **two** biases per unit in each part: 3 x (32 x (8 + 32) + 2 x 32) = **4,032**, fewer than the LSTM's 5,248
- Dense(16): 528, and output: 17
- Total: 4,032 + 528 + 17 = **4,577**

In [ ]:
keras.utils.set_random_seed(42)

gru = keras.Sequential([
    keras.Input(shape=(21, 8)),
    layers.GRU(32),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
gru.summary()

**Compile and train the GRU**, with exactly the same settings as the LSTM, so the comparison is fair.

In [ ]:
gru.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss="binary_crossentropy",
            metrics=[keras.metrics.AUC(name="auc")])

early_stop = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                           patience=5, restore_best_weights=True)

start = time.time()
gru_history = gru.fit(X_seq[idx_tr], y_float[idx_tr],
                      validation_data=(X_seq[idx_val], y_float[idx_val]),
                      epochs=50, batch_size=256,
                      class_weight=class_weight,
                      callbacks=[early_stop], verbose=2)
print(f"Training time: {time.time() - start:.0f} seconds")
print("Epochs trained:", len(gru_history.history["loss"]),
      "| Best epoch:", int(np.argmax(gru_history.history["val_auc"])) + 1)

**Evaluate the GRU on the test users.**

In [ ]:
results.append(evaluate("GRU", y[idx_test], gru.predict(X_seq[idx_test], verbose=0).ravel()))

**Training curves for both networks:** training and validation ROC-AUC for every epoch, numbered from 1, with a dashed line at the best epoch (whose weights each model uses).
`zip(axes, [...])` pairs each of the two charts with one model's history, so the same plotting code runs twice.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, hist) in zip(axes, [("LSTM", lstm_history), ("GRU", gru_history)]):
    epochs = range(1, len(hist.history["auc"]) + 1)
    best_epoch = int(np.argmax(hist.history["val_auc"])) + 1
    ax.plot(epochs, hist.history["auc"], label="Training")
    ax.plot(epochs, hist.history["val_auc"], label="Validation")
    ax.axvline(best_epoch, color="gray", linestyle="--", label="Best epoch")
    ax.set_title(f"{name}: ROC-AUC per epoch")
    ax.set_xlabel("Epoch")
    ax.legend()
plt.show()

*Pending: this explanation is written from the real results, after the first run on the KKBox data.* <!-- PLACEHOLDER_CURVES -->

# Part C: Does the LSTM really use the order of the days?

**The shuffle test.** We mix up the order of each test user's 21 days: the same values, in a random order.
- `rng.permutation(21)` gives the numbers 0 to 20 in a random order, and indexing with it reorders that user's days. Every user gets a different random order. `default_rng(42)` makes the shuffle repeatable.
- The **totals don't change** when we shuffle, so XGBoost on totals would give exactly the same score.
- But the **order** is destroyed: "listened a lot early and stopped recently" can now look like "started recently".

If the LSTM's ROC-AUC **drops clearly**, the LSTM relies on **when** things happened (recency, trend): proof that it learned time patterns.

In [ ]:
rng = np.random.default_rng(42)
X_test_shuffled = X_seq[idx_test].copy()
for i in range(len(X_test_shuffled)):
    X_test_shuffled[i] = X_test_shuffled[i][rng.permutation(21)]

auc_normal = roc_auc_score(y[idx_test], lstm.predict(X_seq[idx_test], verbose=0).ravel())
auc_shuffled = roc_auc_score(y[idx_test], lstm.predict(X_test_shuffled, verbose=0).ravel())
print(f"LSTM ROC-AUC with real day order: {auc_normal:.3f}")
print(f"LSTM ROC-AUC with days shuffled : {auc_shuffled:.3f}")

*Pending: this explanation is written from the real results, after the first run on the KKBox data.* <!-- PLACEHOLDER_SHUFFLE -->

# Part D: Compare all models

**Results table and comparison chart.**
- `pd.DataFrame(results)` turns the list of score dictionaries into a table, saved as `data/kkbox_model_results.csv`.
- This table is **separate** from the Telco table (`data/model_results.csv`), because it's a different dataset and a different problem.
- It is **rebuilt from scratch** on every run (it never reads the old file), so it always has exactly **5 rows**.
- The chart shows Recall, ROC-AUC and PR-AUC for every model except the lazy one.

In [ ]:
results_df = pd.DataFrame(results).round(3)
results_df.to_csv("../data/kkbox_model_results.csv", index=False)

compare = results_df[results_df["Model"] != "Lazy (predict all stay)"]
compare.set_index("Model")[["Recall", "ROC-AUC", "PR-AUC"]].plot(kind="bar", figsize=(10, 5))
plt.title("KKBox: totals vs time features vs sequence models (same test users)")
plt.ylim(0, 1)
plt.xticks(rotation=15, ha="right")
plt.legend(loc="lower right")
plt.show()

results_df

*Pending: this explanation is written from the real results, after the first run on the KKBox data.* <!-- PLACEHOLDER_HONEST -->

*Pending: this explanation is written from the real results, after the first run on the KKBox data.* <!-- PLACEHOLDER_FINDINGS -->

## Why move on (Lesson 9)
We can now predict churn on snapshot data (Telco) and on behavior sequences (KKBox). The last step turns predictions into **action**: rank customers by churn risk and customer value, map segments to retention offers, and serve the model through an API and a dashboard.